In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 20b: Data Insights Documentation Scans for agentic grounding




### NOTE:
This notebook has an interactive authorization step in section 1. You cannot just run the notebook to completion without the interaction.

## 1. Variables/Configs

In [1]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]

PROJECT_NBR_LIST=!gcloud projects describe $PROJECT_ID --format="value(projectNumber)"
PROJECT_NBR=PROJECT_NBR_LIST[0]

LOCATION="us-central1"
DATA_SCAN_API_CREATE_ENDPOINT_PREFIX=f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId="
DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX=(f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/")
SCOPES = ['https://www.googleapis.com/auth/cloud-platform']
BASE_URL_FOR_DATAPLEX_SCAN="https://dataplex.googleapis.com/v1"


print(f"PROJECT_ID: {PROJECT_ID}")
print(f"PROJECT_NBR: {PROJECT_NBR}")
print(f"LOCATION: {PROJECT_ID}")
print(f"DATA_SCAN_API_CREATE_ENDPOINT_PREFIX: {DATA_SCAN_API_CREATE_ENDPOINT_PREFIX}")
print(f"DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX: {DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX}")


PROJECT_ID: data-insights-quickstart
PROJECT_NBR: 606804615020
LOCATION: data-insights-quickstart
DATA_SCAN_API_CREATE_ENDPOINT_PREFIX: https://dataplex.googleapis.com/v1/projects/data-insights-quickstart/locations/us-central1/dataScans?dataScanId=
DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX: https://dataplex.googleapis.com/v1/projects/data-insights-quickstart/locations/us-central1/dataScans/


In [2]:
!gcloud auth application-default login


You are running on a Google Compute Engine virtual machine.
The service credentials associated with this virtual machine
will automatically be used by Application Default
Credentials, so it is not necessary to use this command.

If you decide to proceed anyway, your user credentials may be visible
to others with access to this virtual machine. Are you sure you want
to authenticate with your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=eIf7cYm7qYVkgG0xP8dVXgUvWyiN99&prompt=consent&token_

In [3]:
!pip install google-cloud-dataplex==2.11.0  -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.1/508.1 kB 9.9 MB/s eta 0:00:00


## 2. Data Scan Utils

In [4]:
import requests, json, time, logging
import google.auth
import google.auth.transport.requests
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
from google.api_core.exceptions import NotFound
from google.api_core import exceptions
from urllib.parse import urlencode # Import urlencode



def get_access_token():
    """
    Generates an access token using Application Default Credentials or a service account key file.
    """
    try:
        # Authenticate using Application Default Credentials (ADC)
        # This will automatically find credentials set via `gcloud auth application-default login`
        # or from the environment if running on GCP.
        credentials, project = google.auth.default(scopes=SCOPES)

        # Refresh the credentials to ensure an up-to-date access token
        request = google.auth.transport.requests.Request()
        credentials.refresh(request)
        return credentials.token
    except Exception as e:
        print(f"Error generating access token: {e}")
        return None

def patch_source_table_with_labels(access_token, dataset_id,scan_type, scan_id,  source_table_nm):
    """
    Patches the source BigQuery table with labels to correlate with the data scans

    Args:
        access_token (str): The Google Cloud access token
        dataset_id (str): The dataset id
        scan_type (str):
        scan_id (str): The scan id
        source_table_nm (str): The source table name

    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """
    if not access_token:
            msg="Access token is missing. Cannot proceed with API call."
            return msg

    API_ENDPOINT=f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/datasets/{dataset_id}/tables/{source_table_nm}?"

    patch_request_body= generate_patch_label_request_body(scan_type,scan_id)

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }


    # Apply the patch
    try:
        response = requests.patch(API_ENDPOINT, headers=headers, json=patch_request_body)
        response.raise_for_status()  # Raise an exception for HTTP errors



    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response Body: {response.text}")
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred during the API call: {req_err}")


def generate_scan_request_body(dataset_id,scan_id, scan_type,
                                            source_table_nm, profile_results_table_nm):
    """
    Generates the scan request body

    Args:
        scan_id (str): scan id
        scan_type (str): Type of scan (DATA_PROFLE_SCAN/DATA_DOCUMENTATION_SCAN/DATA_KNOWLEDGE_ENGINE_SCAN)
        source_table_nm (str): Source table name
        profile_results_table_nm (str): The profile results table name


    Returns:
        string: JSON with the request body for the data scan API call
    """
    scan_request_body={}
    if(scan_type == "DATA_PROFILE_SCAN"):

        scan_request_body={
            "displayName": f"{scan_id}",
            "data": {
                "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}/tables/{source_table_nm}"
            },
            "dataProfileSpec": {
                "postScanActions":
                {
                    "bigqueryExport":
                    {
                        "resultsTable": f"projects/{PROJECT_ID}/datasets/{SCAN_RESULTS_BQ_DATASET_ID}/tables/{profile_results_table_nm}"
                    }
                    }

            },
            "executionSpec": {
                "trigger": {
                    "onDemand": {} # Run on demand for this example
                }
            }
        }
    elif(scan_type == "DATA_DOCUMENTATION_SCAN"):
        scan_request_body={
        "displayName": f"{scan_id}",
        "type": "DATA_DOCUMENTATION",
        "dataDocumentationSpec": {},
        "data": {
            "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}/tables/{source_table_nm}"
        },
        "executionSpec": {
            "trigger": {
                "onDemand": {} # Run on demand for this example
            }
        }
    }
    elif(scan_type == "DATA_KNOWLEDGE_ENGINE_SCAN"):
        scan_request_body={
        "displayName": f"{scan_id}",
        "type": "DATA_DOCUMENTATION",
        "dataDocumentationSpec": {},
        "dataDocumentationResult": {},
        "data": {
            "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}"
        },
        "executionSpec": {
            "trigger": {
                "onDemand": {} # Run on demand for this example
            }
        },

    }

    return scan_request_body

def generate_patch_label_request_body(scan_type, scan_id):
    """
        Returns the patch labels json that needs to be attached to the source table to tie programmatic scans to the UI

        Args:
            scan_type (str): Type of scan (DATA_PROFLE_SCAN/DATA_DOCUMENTATION_SCAN/DATA_KNOWLEDGE_ENGINE_SCAN)
            scan_id (str): Scan id
            operation_type (str): Type of operation (CREATE_SCAN/RUN_SCAN)

        Returns:
            string: json with the patch labels
        """
    label_json=""
    scan_stub = ""
    if scan_type == "DATA_PROFILE_SCAN":
        scan_stub="dp"
    elif scan_type == "DATA_DOCUMENTATION_SCAN":
        scan_stub="data-documentation"
    elif scan_type == "DATA_KNOWLEDGE_ENGINE_SCAN":
        scan_stub="data-documentation"


    label_json = {
        "labels": {f"dataplex-{scan_stub}-published-scan":f"{scan_id}",
                 f"dataplex-{scan_stub}-published-project":f"{PROJECT_ID}",
                 f"dataplex-{scan_stub}-published-location":f"{LOCATION}"}
      }

    return label_json


def get_scan_api_endpoint(scan_operation_type,scan_id):
    """
    Returns the scan API endpoint

    Args:
        scan_operation_type (str): Type of operation (CREATE_SCAN/RUN_SCAN)
        scan_id (str): Scan id

    Returns:
        string: API endpoint
    """
    scan_api_endpoint=""

    if(scan_operation_type == "CREATE_SCAN"):
        scan_api_endpoint=f"{DATA_SCAN_API_CREATE_ENDPOINT_PREFIX}{scan_id}"
    elif (scan_operation_type == "LIST_SCAN"):
        scan_api_endpoint=f"{DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX}{scan_id}"
    else:  # run scan
        scan_api_endpoint=f"{DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX}{scan_id}:run"


    return scan_api_endpoint

def check_if_scan_already_exists(access_token,scan_id):
    """
    Calls the BigQuery Scan API to check if a Data Scan already exists.

    Args:
        access_token (str): token
        scan_api_endpoint (str): API endpoint
        scan_id (str): Scan name

    Returns:
        string: NOT_FOUND or EXISTS_ALREADY
    """


    if not access_token:
        print("Access token is missing. Cannot proceed with API call.")
        return

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }


    request_body = {}
    scan_list_api_endpoint= DATA_SCAN_API_EXECUTION_ENDPOINT_PREFIX + scan_id


    try:
        response = requests.get(scan_list_api_endpoint, headers=headers, json=request_body)
        response.raise_for_status()  # Raise an exception for HTTP errors
        response_json = response.json()

        print(f"\nAPI Call Successful! Status Code: {response.status_code}")


        # Check for status node - if it is found, it says  "status": "NOT_FOUND" - it means the scan does not exist
        not_found = response_json['status']

        if not_found:
            return "NOT_FOUND"
        else:
            return "EXISTS_ALREADY"

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response Body: {response.text}")
        return "NOT_FOUND"
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred during the API call: {req_err}")
        return "NOT_FOUND2"



def create_scan_synchronous(access_token,dataset_id, scan_id, scan_type, source_table_nm, scan_results_table_nm):
    """
    Calls the BigQuery Scan API with the generated access token.

    Args:
        access_token (str): token
        scan_id (str): Scan id
        scan_type (str): DATA_PROFILE_SCAN/DATA_DOCUMENTATION_SCAN/DATA_KNOWLEDGE_ENGINE_SCAN
        source_table_nm (str): Name of source table in BQ
        scan_results_table_nm (str): Name of profile results table in BQ

    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """
    if not access_token:
        print("Access token is missing. Cannot proceed with API call.")
        return

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    #scan_list_status = check_if_scan_already_exists(access_token, scan_id)
    scan_list_status = "NOT_FOUND"

    if scan_list_status == "NOT_FOUND":

        scan_request_body = generate_scan_request_body(dataset_id,scan_id, scan_type, source_table_nm, scan_results_table_nm)
        scan_create_api_endpoint = get_scan_api_endpoint("CREATE_SCAN",scan_id)
        print(f"\nCalling API: {scan_create_api_endpoint}")
        print(f"Request Body: {json.dumps(scan_request_body, indent=2)}")

        try:
            # The ':run' method is a POST request. It returns a Long Running Operation (LRO).
            response = requests.post(scan_create_api_endpoint, headers=headers, json=scan_request_body)
            response.raise_for_status()  # Raise an exception for HTTP errors
            initial_response = response.json()
            # Extract the operation name to poll
            operation_name = initial_response.get("name")
            if operation_name:

                # Poll for the completion of the operation
                final_result = poll_data_scan_operation("CREATE_SCAN",operation_name, access_token)
                if final_result:
                    print(f"Scan successfully completed and results obtained. Operation name: {operation_name}")
                else:
                    print(f"Scan operation did not complete successfully or timed out. Operation name: {operation_name}")
            else:
                print("Could not find 'name' in the initial response. Cannot poll for completion.")

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred: {http_err}")
            print(f"Response Body: {response.text}")
        except requests.exceptions.RequestException as req_err:
            print(f"An error occurred during the API call: {req_err}")
    else:
        print("Scan already exists; Skipping creation..")

def poll_data_scan_operation(operation_type: str, operation_name: str, access_token: str, poll_interval_seconds=30, timeout_minutes=30):
    """
    Polls the Dataplex Operation API to check for the completion of a data scan.

    Args:
        operation_name (str): The full resource name of the LRO (Long Running Operation)
                              returned by the data scan run API call.
        access_token (str): The Google Cloud access token.
        poll_interval_seconds (int): How often to poll the API, in seconds.
        timeout_minutes (int): Maximum time to wait for the operation to complete, in minutes.

    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    # Base URL for Google Cloud Long Running Operations API
    # Example operation_name: projects/PROJECT_ID/locations/LOCATION/operations/OPERATION_ID
    # We need to ensure the base URL matches how the operation_name is structured for the API call.
    # The operation_name already contains the full path, so we use it directly.
    operation_api_url = f"https://dataplex.googleapis.com/v1/{operation_name}"

    print(f"\nPolling operation: {operation_name}")
    start_time = time.time()

    while (time.time() - start_time) < (timeout_minutes * 60):
        try:
            response = requests.get(operation_api_url, headers=headers)
            response.raise_for_status()
            operation_status = response.json()


            if(operation_type == "CREATE_SCAN"):
                if operation_status.get("done"):
                    print(f"Operation {operation_name} completed.")
                    if "error" in operation_status:
                        print(f"Operation failed with error: {operation_status['error']}")
                        return None
                    elif "response" in operation_status:
                        print(f"Operation succeeded. Result: {json.dumps(operation_status['response'], indent=2)}")
                        return operation_status["response"]
                    else:
                        print("Operation finished, but no explicit response or error found.")
                        return operation_status # Return the full status for further inspection
                else:
                    print(f"Operation {operation_name} still running. Retrying in {poll_interval_seconds} seconds...")
                    time.sleep(poll_interval_seconds)

            else:
                if operation_status["state"].upper()=="COMPLETED" or operation_status["state"].upper()=="SUCCEEDED" or operation_status["state"].upper()=="DONE":
                    print(f"Operation {operation_name} completed.")
                    if "error" in operation_status:
                        print(f"Operation failed with error: {operation_status['error']}")
                        return None
                    elif "response" in operation_status:
                        print(f"Operation succeeded. Result: {json.dumps(operation_status['response'], indent=2)}")
                        return operation_status["response"]
                    else:
                        print("Operation finished, but no explicit response or error found.")
                        return operation_status # Return the full status for further inspection
                else:
                    print(f"Operation {operation_name} still running. Retrying in {poll_interval_seconds} seconds...")
                    time.sleep(poll_interval_seconds)


        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred during polling: {http_err}")
            print(f"Response Body: {response.text}")
            return None
        except requests.exceptions.RequestException as req_err:
            print(f"An error occurred during polling the operation: {req_err}")
            return None

    print(f"Polling timed out after {timeout_minutes} minutes for operation: {operation_name}")
    return None


def run_scan_synchronous(access_token, scan_id):
    """
    Calls the Dataplex API to run a previously created Scan and polls for its completion.

    Args:
        access_token (str): token
        scan_api_endpoint (str): API endpoint
        scan_name (str): Name of the precreated scan

    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """

    if not access_token:
        print("Access token is missing. Cannot proceed with API call.")
        return

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    # The request body for a ':run' operation is typically empty for on-demand execution.
    request_body = {}

    scan_run_api_endpoint = get_scan_api_endpoint("RUN_SCAN",scan_id)

    print(f"\nAttempting to run scan: {scan_id} at {scan_run_api_endpoint}")


    try:
        # The ':run' method is a POST request. It returns a Long Running Operation (LRO).
        response = requests.post(scan_run_api_endpoint, headers=headers, json=request_body)
        response.raise_for_status()  # Raise an exception for HTTP errors
        initial_response = response.json()

        # Extract the operation name to poll
        operation_name = initial_response['job']['name']
        #initial_response.get("job.name")
        if operation_name:
            # Poll for the completion of the operation
            final_result = poll_data_scan_operation("RUN_SCAN",operation_name, access_token)
            if final_result:
                print(f"\nScan successfully completed and results obtained. Operation name: {operation_name}")
                print(final_result)
            else:
                print(f"\nScan operation did not complete successfully or timed out. Operation name: {operation_name}")
        else:
            print("Could not find 'name' in the initial response. Cannot poll for completion.")

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response Body: {response.text}")
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred during the API call: {req_err}")

def run_scan_async(access_token, scan_id):
    """
    Calls the Dataplex API to run a previously created Scan and polls for its completion.

    Args:
        access_token (str): token
        scan_api_endpoint (str): API endpoint
        scan_name (str): Name of the precreated scan

    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """

    if not access_token:
        print("Access token is missing. Cannot proceed with API call.")
        return

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    # The request body for a ':run' operation is typically empty for on-demand execution.
    request_body = {}

    scan_run_api_endpoint = get_scan_api_endpoint("RUN_SCAN",scan_id)

    print(f"\nAttempting to run scan: {scan_id} at {scan_run_api_endpoint}")


    try:
        # The ':run' method is a POST request. It returns a Long Running Operation (LRO).
        response = requests.post(scan_run_api_endpoint, headers=headers, json=request_body)
        response.raise_for_status()  # Raise an exception for HTTP errors
        initial_response = response.json()
        print(f"initial_response: {initial_response}")

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response Body: {response.text}")
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred during the API call: {req_err}")


def list_scans(access_token, scan_type):
    """
    Calls the Dataplex API to list data scans, optionally filtering by type.

    Args:
        access_token (str): token
        scan_type (str): DATA_PROFILE_SCAN/DATA_DOCUMENTATION_SCAN/DATA_KNOWLEDGE_ENGINE_SCAN

    Returns:
        str: A formatted string of the scan list if successful, or an error
             message string.
    """
    if not access_token:
        msg ="Access token is missing. Cannot proceed with API call."
        print(msg)
        return msg

    headers = {
        "Authorization": f"Bearer {access_token}",
    }

    scan_list_api_endpoint = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans"


    scan_type_map = {
        "DATA_PROFILE_SCAN": "DATA_PROFILE",
        "DATA_DOCUMENTATION_SCAN": "DATA_DOCUMENTATION",
        "DATA_KNOWLEDGE_ENGINE_SCAN": "KNOWLEDGE_ENGINE"
    }
    params = {}


    if scan_type != "ALL":
       api_scan_type = scan_type_map.get(scan_type)
       if api_scan_type:
           params['filter'] = f'type="{api_scan_type}"'
       scan_list_api_endpoint = f"{scan_list_api_endpoint}?{urlencode(params)}"

       if api_scan_type == scan_type_map.get("DATA_KNOWLEDGE_ENGINE_SCAN"):
        markdown_table = "| Dataset |  Scan |  State | \n"
        markdown_table += "|---|---|---|\n"
       else:
        markdown_table = "| Dataset | Table | Scan |  State | \n"
        markdown_table += "|---|---|---|---|\n"



    try:
        response = requests.get(scan_list_api_endpoint, headers=headers)
        response.raise_for_status()  # Raises an error for bad status codes (4xx or 5xx)
        response_json = response.json()

        if "dataScans" in response_json and response_json["dataScans"]:
            for scan in response_json["dataScans"]:
                table_resource_uri = scan.get('data', {}).get('resource', '')
                table_resource_uri_parts = table_resource_uri.split("/")
                source_dataset_id = table_resource_uri_parts[6]

                if(source_dataset_id in SOURCE_BQ_DATASETS_IN_SCOPE):

                    if api_scan_type == scan_type_map.get("DATA_KNOWLEDGE_ENGINE_SCAN"):
                        markdown_table += (f"| {source_dataset_id} |  {scan.get('displayName', 'N/A')} |  {scan.get('state', 'N/A')} | \n")
                    else:

                        if len(table_resource_uri_parts) >= 8:

                            source_table_id = table_resource_uri_parts[8]
                            markdown_table += (f"| {source_dataset_id} | {source_table_id} | {scan.get('displayName', 'N/A')} |  {scan.get('state', 'N/A')} | \n")

            return "\n".join(markdown_table)
        else:
            print("No data scans found in this location.")
            return "No data scans found in this location."


    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code} - {e.response.text}")
        error_message = f"HTTP Error: {e.response.status_code} - {e.response.text}"
        return error_message
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        error_message = f"An unexpected error occurred: {e}"
        return error_message

def list_scan_jobs(access_token: str, scan_id: str):
    """
    Calls the Dataplex API to list data scan jobs, optionally filtering by type.

    Args:
        access_token (str): token
        scan_id (str): scan name

    Returns:
        str: A formatted string of the scan job list in table markdown format if successful, or an error
             message string.
    """
    if not access_token:
        msg ="Access token is missing. Cannot proceed with API call."
        print(msg)
        return msg

    headers = {
        "Authorization": f"Bearer {access_token}",
    }

    scan_job_list_api_endpoint = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}/jobs"

    try:
        response = requests.get(scan_job_list_api_endpoint, headers=headers)
        response.raise_for_status()  # Raises an error for bad status codes (4xx or 5xx)
        response_json = response.json()

        markdown_table = "| Job_Name | UID | State | Start_Time | End_Time |\n"
        markdown_table += "|---|---|---|---|---|\n"


        if "dataScanJobs" in response_json and response_json["dataScanJobs"]:
            for job in response_json["dataScanJobs"]:
                markdown_table += (f"| {job['name'].split('/')[-1]} | {job.get('uid', 'N/A')} | {job.get('state', 'N/A')} |  {job.get('startTime', 'N/A')} | {job.get('endTime', 'N/A')} | \n")

            return "\n".join(markdown_table)

        else:
            print("No jobs found for this data scan.")


    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code} - {e.response.text}")
        error_message = f"HTTP Error: {e.response.status_code} - {e.response.text}"
        return error_message
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        error_message = f"An unexpected error occurred: {e}"
        return error_message


def fetch_scan_results(access_token: str, scan_id: str):
    """
    Calls the Dataplex API to list data scan results.

    Args:
        access_token (str): token
        scan_id (str): scan name

    Returns:
        str: A formatted string of the results including markdown tables where applicable
    """
    if not access_token:
        msg ="Access token is missing. Cannot proceed with API call."
        print(msg)
        return msg

    headers = {
        "Authorization": f"Bearer {access_token}",
    }

    scan_results_list_api_endpoint = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}?view=FULL"


    try:
        response = requests.get(scan_results_list_api_endpoint, headers=headers)
        response.raise_for_status()  # Raises an error for bad status codes (4xx or 5xx)
        response_json = response.json()
        return response_json

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code} - {e.response.text}")
        try:
            # Return JSON error if available, otherwise raw text
            return e.response.json()
        except json.JSONDecodeError:
            print(f"Failed to decode JSON from response: {e}")
            return {"error": e.response.text}
    except json.JSONDecodeError as e:
        print(f"Failed to decode JSON from response: {e}")
        return {"error": f"Failed to decode JSON from response: {e}"}
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return {"error": f"An unexpected error occurred: {e}"}


def persist_documentation_scan_table_metadata(access_token: str, scan_id: str):
    """
    Persists the table metadata generated by the data documentation scan .

    Args:
        access_token: token
        scan_id: scan id

    Returns:
        A string indicating the status
    """
    if not access_token:
        msg ="Access token is missing. Cannot proceed with API call."
        print(msg)
        return msg

    scan_result = fetch_scan_results(access_token, scan_id)

    # Extract info from scan result
    table_resource_uri = scan_result["data"]["resource"]
    updated_table_description = scan_result["dataDocumentationResult"]["tableResult"]["overview"]
    updated_schema_with_column_descriptions = scan_result["dataDocumentationResult"]["tableResult"]["schema"]

    # Check for errors from the API call
    if not isinstance(scan_result, dict) or "error" in scan_result:
        print(f"Error fetching scan results: {scan_result}")
        return f"Failed to fetch or parse scan results: {scan_result}"

    try:

        # Parse table URI
        # e.g., //bigquery.googleapis.com/projects/PROJECT_ID/datasets/DATASET_ID/tables/TABLE_ID
        parts = table_resource_uri.split("/")
        project_id = parts[4]
        dataset_id = parts[6]
        table_id = parts[8]

        # Get BQ client and table
        bq_client = get_bq_client()
        if not bq_client:
            return "Failed to get BigQuery client."

        table_ref = bq_client.dataset(dataset_id, project=project_id).table(
            table_id
        )
        table = bq_client.get_table(table_ref)


        # Update table description
        table.description = updated_table_description
        existing_table_schema = table.schema


        # Update column descriptions by creating a new schema
        updated_schema = []
        for existing_field in existing_table_schema:

            for item in updated_schema_with_column_descriptions["fields"]:
                if item["name"] == existing_field.name:
                    updated_field = bigquery.SchemaField(
                    existing_field.name,
                    existing_field.field_type,
                    existing_field.mode,
                    description=item["description"],
                    )
                    updated_schema.append(updated_field)



        table.schema = updated_schema

        # Update the table
        bq_client.update_table(table, ["description", "schema"])

        print(
            "Successfully updated metadata for table "
            f"{project_id}.{dataset_id}.{table_id}"
        )
        return "Succeeded"

    except (KeyError, IndexError) as e:
        print(f"Error parsing scan result: {e}")
        return f"Failed to parse scan result: {e}"
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return f"An unexpected error occurred: {e}"


def patch_source_dataset_with_labels(access_token, dataset_id,scan_type, scan_id):
    """
    Patches the source BigQuery dataset with labels to correlate with the data scans

    Args:
        access_token (str): The Google Cloud access token
        dataset_id (str): The dataset id
        scan_type (str):
        scan_id (str): The scan id


    Returns:
        dict: The final operation response if successful, None if timed out or failed.
    """
    if not access_token:
            msg="Access token is missing. Cannot proceed with API call."
            return msg

    API_ENDPOINT=f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/datasets/{dataset_id}?"

    patch_request_body= generate_patch_label_request_body(scan_type,scan_id)

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }


    # Apply the patch
    try:
        response = requests.patch(API_ENDPOINT, headers=headers, json=patch_request_body)
        response.raise_for_status()  # Raise an exception for HTTP errors



    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response Body: {response.text}")
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred during the API call: {req_err}")

def execute_table_documentation_scan_for_a_dataset(DATASET_ID: str) -> str:

  status_string=""

  try:

      # 1. Generate the access token
      token = get_access_token()

      if token:
          status_string += "Successfully generated access token."
          # Run the scan
          status_string += "\nStarting the scan for the entire dataset....."

          dataset_id=DATASET_ID.lower()

          scan_type="DATA_DOCUMENTATION_SCAN"

          # 2. Create & run the scan
          table_list = fetch_list_of_tables_in_dataset(dataset_id)
          for source_table_nm in table_list:
            print("=============================================")
            print(source_table_nm)
            print("=============================================")

            # 2a. Create the scan (does not automatically execute the scan)
            scan_id = 'fridge-' + source_table_nm.replace("_","-") + "-table-documentation-scan"
            create_scan_synchronous(token,dataset_id, scan_id,scan_type,source_table_nm,"")
            status_string += f"Successfully created (if it didnt exist) the scan: {scan_id}"

            # 2b. Run the patch below so that programmatically executed scans show up in the Cloud Console UI
            patch_source_table_with_labels(token, dataset_id, scan_type, scan_id, source_table_nm)
            status_string += f"Successfully patched source tables with the scan name: {scan_id}"

            # 2c. Executes the scan
            run_scan_synchronous(token, scan_id)
            status_string += f"Successfully ran the scan: {scan_id}"

            # 2d. Persist the scan results to the table
            persist_documentation_scan_table_metadata(token,scan_id)
            status_string += f"Successfully persisted the scan results: {scan_id}"


          return status_string

      else:
        return "ERROR: Failed to get access token. Please check your credentials and permissions."

  except Exception as e:
    return f"ERROR: Failed to run table documentation scans. Specific error message is: {str(e)}"


def execute_dataset_documentation_scan(DATASET_ID: str) -> str:


  try:
    # 1. Generate the access token
    token = get_access_token()

    if token:

        dataset_id=DATASET_ID.lower()
        scan_type="DATA_KNOWLEDGE_ENGINE_SCAN"


        # 2a. Create the scan (does not automatically execute the scan)
        scan_id = dataset_id.replace("_","-") + "-dataset-documentation-scan"
        create_scan_synchronous(token,dataset_id, scan_id,scan_type,"","")


        # 2b. Run the patch below so that programmatically executed scans show up in the Cloud Console UI
        patch_source_dataset_with_labels(token, dataset_id, scan_type, scan_id)


        # 2c. Execute the scan
        run_scan_synchronous(token, scan_id)


        return f"Successfully ran the scan: {scan_id}"

    else:
      return "ERROR: Failed to get access token. Please check your credentials and permissions."

  except Exception as e:
    return f"ERROR: Failed to run dataset documentation scan. Specific error message is: {str(e)}"

## 3. BQ Utils

In [5]:
from google.cloud import bigquery
from google.api_core import exceptions
import google.auth
import itertools, json, logging
from typing import Optional


def get_bq_client() -> Optional[bigquery.Client]:
  """Initializes and returns a BigQuery client.

  Returns:
      Optional[bigquery.Client]: A BigQuery client instance, or None if
      initialization fails.
  """

  try:
    client = bigquery.Client(project=PROJECT_ID)
    return client
  except google.auth.exceptions.DefaultCredentialsError as e:
    print(f"Authentication failed: {e}")
    print(
        "Please configure your GCP credentials."
        "See https://cloud.google.com/docs/authentication/provide-credentials-adc"
    )
    return None
  except Exception as e:
    print(f"An unexpected error occurred while creating BigQuery client: {e}")
    return None


def execute_bq_sql_query(
    sql_query: str
) -> Optional[bigquery.table.RowIterator]:
    """Executes a SQL query and returns the results.

    Args:
        bq_client: The BigQuery client.
        sql_query: The SQL query to execute.

    Returns:
        Optional[bigquery.table.RowIterator]: An iterator for the query
        results, or None if an error occurs.
    """

    try:
        bq_client = get_bq_client()

        if not bq_client:
            print("BigQuery client is not available.")
            return None

        rows = bq_client.query_and_wait(sql_query)  # Make an API request.
        return rows
    except exceptions.GoogleAPICallError as e:
        print(f"BigQuery API call failed: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def generate_markdown_table_from_bq_rows(row_iterator: bigquery.table.RowIterator):
    """Generates a Markdown table from a BigQuery row iterator."""


    if not row_iterator:
        return "No results to display."
    try:
        headers = [field.name for field in row_iterator.schema]

        markdown_table = "|" + "|".join(headers) + "|\n"
        markdown_table += "|" + "|".join(["---"] * len(headers)) + "|\n"

        for row in row_iterator:
            row_values = [str(row[header]) for header in headers]
            markdown_table += "|" + "|".join(row_values) + "|\n"

        print(f"Markdown Table: {markdown_table}")

        return markdown_table
    except Exception as e:
        print(f"Error generating markdown table: {e}")
        return "Error generating markdown table."

def get_query_results_markdown(sql_query: str) -> str:
    """Executes a SQL query and returns the results as a Markdown table."""

    rows = execute_bq_sql_query(sql_query)
    if rows is None:
        print("Failed to execute query and retrieve results.")
        return "Failed to execute query and retrieve results."
    else:
        markdown_table = generate_markdown_table_from_bq_rows(rows)
        return markdown_table

def field_to_dict(field: bigquery.SchemaField) -> dict:
        """
        Recursively convert a SchemaField into a dict, including subfields if any.
        """
        field_dict = {"name": field.name, "description": field.description or ""}
        # If the field is a RECORD with nested fields, recurse
        if field.fields:
            field_dict["fields"] = [
                field_to_dict(subfield) for subfield in field.fields
            ]
        return field_dict

def fetch_all_tables_metadata_json() -> str:
    """
    Retrieves detailed info about each table in the specified datasets and
    returns it as a JSON string. For each table, it includes:
      - table_name (full path in 'project.dataset.table')
      - description
      - columns (list of columns with name, description)
        * handles nested fields (RECORD type) recursively
    """

    table_iterators = []
    project_id = PROJECT_ID

    try:
        bq_client = bigquery.Client(project=project_id)
        bq_dataset_list = SOURCE_BQ_DATASETS_IN_SCOPE

        for ds_id in bq_dataset_list:

            try:
                table_iterators.append(bq_client.list_tables(f"{project_id}.{ds_id}"))
            except exceptions.NotFound:
                print(f"Dataset not found, skipping: {project_id}.{ds_id}")
    except google.auth.exceptions.DefaultCredentialsError as e:
        print(f"Authentication failed: {e}")
        return json.dumps({"error": f"Authentication failed: {e}"})
    except Exception as e:
        print(f"An unexpected error occurred during client setup: {e}")
        return json.dumps(
            {"error": f"An unexpected error occurred during client setup: {e}"}
        )

    all_tables_info = []
    for table_item in itertools.chain.from_iterable(table_iterators):
        full_table_id = ""  # Initialize here for the except block
        try:
            full_table_id = (
                f"{table_item.project}.{table_item.dataset_id}.{table_item.table_id}"
            )

            table_obj = bq_client.get_table(full_table_id)
            table_info = {
                "table_name": full_table_id,
                "description": table_obj.description or "",
                "columns": [field_to_dict(f) for f in table_obj.schema],
            }
            all_tables_info.append(table_info)
        except exceptions.NotFound:
            print(f"Table not found, skipping: {full_table_id}")
            continue
        except Exception as e:
            print(f"Could not process table {full_table_id}: {e}")
            continue

    # Convert the list of table dictionaries to a JSON string
    return json.dumps(all_tables_info, indent=2)

def fetch_bq_table_schema(table_fq_resource_uri: str) -> str:
    """
    Retrieves the BQ table metadata
    """

    try:
        bq_client = bigquery.Client(project=PROJECT_ID)
        table_resource_uri_parts = table_fq_resource_uri.split("/")
        full_table_id=table_resource_uri_parts[6] + "." + table_resource_uri_parts[8].rstrip('"')
        try:
            table_obj = bq_client.get_table(full_table_id.strip())


            serializable_schema = []
            for field in table_obj.schema:
                field_dict = {
                    "name": field.name,
                    "type": field.field_type,
                    "mode": field.mode,
                    "description": field.description,
                }
                if field.fields:  # Handle nested fields for RECORD types
                    field_dict["fields"] = [
                        {
                            "name": nested_field.name,
                            "type": nested_field.field_type,
                            "mode": nested_field.mode,
                            "description": nested_field.description,
                        }
                        for nested_field in field.fields
                    ]
                serializable_schema.append(field_dict)

            return json.dumps(serializable_schema, indent=2)
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return json.dumps({"error": f"An unexpected error occurred: {e}"}
        )

    except google.auth.exceptions.DefaultCredentialsError as e:
        print(f"Authentication failed: {e}")
        return json.dumps({"error": f"Authentication failed: {e}"})
    except exceptions.NotFound:
        print(f"Resource not found: {table_fq_resource_uri}")
        return json.dumps(
            {"error": "Resource not found"}
        )
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return json.dumps(
            {"error": f"An unexpected error occurred: {e}"}
        )

def update_table_schema(project_id: str, dataset_id: str, table_id: str, new_schema: list[bigquery.SchemaField]):
    """
    Updates the schema of a BigQuery table.

    Args:
        project_id: The project ID.
        dataset_id: The dataset ID.
        table_id: The table ID.
        new_schema: A list of bigquery.SchemaField objects for the new schema.
    """

    # Instantiates a BQ connection
    try:
        bq_client = bigquery.Client(project=project_id)
        print("BigQuery client initialized successfully.")
    except Exception as e:
        print(f"Error initializing BigQuery client: {e}")
        print("Please ensure you have authenticated with 'gcloud auth application-default login'")
        print("and that your PROJECT_ID is correct.")
        return

    table_ref = bq_client.dataset(dataset_id).table(table_id)

    try:
        table = bq_client.get_table(table_ref)
        print(f"Fetched table: {table.project}.{table.dataset_id}.{table.table_id}")
    except NotFound:
        print(f"Error: Table {dataset_id}.{table_id} not found.")
        return
    except Exception as e:
        print(f"An error occurred while fetching the table: {e}")
        return

    # Set the table's schema to the newly constructed schema.
    table.schema = new_schema

    # Make the API call to update the table's schema.
    # The second argument to update_table() specifies which properties to update.
    try:
        table = bq_client.update_table(table, ["schema"])  # API request
        print(f"\nSuccessfully updated the table schema for {table.table_id}.")

        #for field in table.schema:
        #    print(f" - {field.name} ({field.field_type}): {field.description}")
    except Exception as e:
        print(f"\nAn error occurred while updating the table schema: {e}")


def fetch_list_of_tables_in_dataset(dataset_id: str) :
    """Fetches a list of table IDs in a given BigQuery dataset.

    Args:
        dataset_id: The ID of the dataset.

    Returns:
        A list of table IDs, or an empty list if an error occurs.
    """
    client = get_bq_client()
    if not client:
        print("Failed to initialize BigQuery client.")
        return ["Failed to initialize BigQuery client."]

    try:
        bq_tables_list = client.list_tables(dataset_id)  # Make an API request.
        return [table.table_id for table in bq_tables_list]
    except exceptions.NotFound:
        print(f"Dataset not found: {dataset_id}")
        return ["Dataset not found."]

    except Exception as e:
        print(f"An unexpected error occurred while listing tables: {e}")
        return ["An unexpected error occurred while listing tables."]



## 4. Utils for agentic grounding file generation

In [6]:
import pandas as pd
from typing import List, Dict, Any, Optional
import google.auth
import re
from google.cloud import bigquery
from google.cloud import storage

def get_auth_token():
    creds, _ = google.auth.default()
    # Refresh the credentials to get an access token
    creds.refresh(google.auth.transport.requests.Request())
    return creds.token

def sanitize_string_with_hyphens(input_string):
    """
    Converts a string to lowercase and replaces any character that is not
    a lowercase letter or a number with a hyphen.
    """
    # convert the entire string to lowercase.
    processed_string = input_string.lower()

    # The pattern [^a-z0-9] matches any single character that is NOT
    # a lowercase letter (a-z) or a digit (0-9).
    sanitized_string = re.sub(r'[^a-z0-9]', '-', processed_string)

    return sanitized_string

def truncate_bigquery_table(bq_table_uri: str):
    """
    Deletes all rows from a specified BigQuery table.

    This function executes a TRUNCATE TABLE DML statement, which is an
    efficient way to clear a table.

    Returns:
        A string confirming the successful truncation or describing an error.
    """
    try:
        # Construct a BigQuery client object.
        client = bigquery.Client()

        # Sanitize the table URI by wrapping it in backticks.
        safe_table_uri = f"`{bq_table_uri}`"

        # Construct the DML query to truncate the table.
        truncate_query = f"TRUNCATE TABLE {safe_table_uri}"

        # Execute the query.
        print(f"Executing query: {truncate_query}")
        query_job = client.query(truncate_query)

        # Wait for the DML query to complete.
        query_job.result()

        return f"Successfully truncated table {bq_table_uri}"

    except exceptions.NotFound:
        return f"Error: The table '{bq_table_uri}' was not found."
    except Exception as e:
        return f"An unexpected error occurred: {e}"

def read_bigquery_table(
    project_id: str,
    dataset_id: str,
    table_id: str
) -> pd.DataFrame:
    """
    Reads all rows from a BigQuery table and returns them as a pandas DataFrame.

    This function authenticates using the environment's default credentials and uses
    the BigQuery Storage Read API for efficient data retrieval.

    Returns:
        A pandas DataFrame containing all rows from the table.
        Returns an empty DataFrame if the table is not found or an error occurs.
    """
    # Construct a BigQuery client object.
    # The client library will automatically handle authentication.
    client = bigquery.Client(project=project_id)

    # Construct the full table ID in the format `project.dataset.table`.
    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    try:
        # Use the list_rows() method to get a row iterator from the API.
        # The to_dataframe() method downloads all rows and converts them to a DataFrame.
        print(f"Reading all rows from table: {table_ref}...")
        rows = client.list_rows(table_ref)
        dataframe = rows.to_dataframe()
        print(f"Successfully read {len(dataframe)} rows.")
        return dataframe

    except Exception as e:
        print(f"An error occurred: {e}")
        # Return an empty DataFrame in case of an error.
        return pd.DataFrame()

def write_dict_to_bigquery(bq_table_uri: str, data_to_insert: dict, delete_conditions: dict = None):
    """
    Writes a row of data to a BigQuery table.

    Optionally deletes rows from the table based on a condition before inserting new data.
    Create the dataset if it doesn't exist.
    Create the table if it doesn't exist, using the data's keys as the schema.
    Truncate the table if it exists.
    Update the table schema if the data contains new columns.
    Insert the data as a new row.

    Returns:
        A string indicating success or failure.
    """
    if not data_to_insert:
        return "No data provided to write to BigQuery."

    try:
        # Construct a BigQuery client object.
        client = bigquery.Client()

        # Parse the table URI.
        project_id, dataset_id, table_id = bq_table_uri.split('.')
        dataset_ref = client.dataset(dataset_id)
        table_ref = dataset_ref.table(table_id)
        table_ref_str = f"{project_id}.{dataset_id}.{table_id}"

        # Create dataset if it doesn't exist.
        try:
            client.get_dataset(dataset_ref)
        except exceptions.NotFound:
            print(f"Dataset {dataset_id} not found. Creating it in us-central1.")
            dataset = bigquery.Dataset(dataset_ref)
            dataset.location = "us-central1"
            client.create_dataset(dataset, exists_ok=True)

        # Prepare the schema based on the data to insert.
        new_schema_fields = [
            bigquery.SchemaField(key, "STRING") for key in data_to_insert
        ]

        # Check if table exists and update schema if necessary.
        try:
            table = client.get_table(table_ref)
            current_schema = {field.name for field in table.schema}
            new_schema = list(table.schema)

            for field in new_schema_fields:
                if field.name not in current_schema:
                    new_schema.append(field)

            table.schema = new_schema
            client.update_table(table, ["schema"])

        except exceptions.NotFound:
            print(f"Table {table_id} not found. Creating it.")
            table = bigquery.Table(table_ref_str, schema=new_schema_fields)
            client.create_table(table)

        # Delete rows if conditions are provided.
        if delete_conditions and table_id in ["tables", "columns"]:
            where_clauses = []
            for condition in delete_conditions:
                column = condition.get('column')
                value = condition.get('value')
                if column and value is not None:
                    where_clauses.append(f"{column} = '{value}'")

            if where_clauses:
                delete_query = f"DELETE FROM {table_ref_str} WHERE " + " AND ".join(where_clauses)
                query_job = client.query(delete_query)
                query_job.result()  # Wait for the job to complete.

        # Insert the new row.
        row_to_insert = {key: str(value) for key, value in data_to_insert.items()}
        errors = client.insert_rows_json(table_ref_str, [row_to_insert])

        if not errors:
            return f"Successfully wrote data to {table_ref_str}"
        else:
            return f"Failed to insert rows: {errors}"

    except Exception as e:
        return f"An unexpected error occurred: {e}"

def describe_dataset(description_df: pd.DataFrame) -> str:
    """
    Converts the dataset description DataFrame into a human-readable string.

    Returns:
        A string containing the dataset's description.
    """
    try:
        # Extracts the first description from the DataFrame
        description = description_df["dataset_description"].iloc[0]
        return f"Dataset Overview:\n{description}"
    except (IndexError, KeyError):
        return "No dataset description was found."

def describe_relationships(relationships_df: pd.DataFrame) -> List[str]:
    """
    Converts the relationships DataFrame into human-readable sentences.

    Returns:
        A list of strings, where each string describes a table relationship.
    """
    descriptions = []
    for _, row in relationships_df.iterrows():
        # Constructs a sentence describing the join between two tables
        sentence = (
            f"Table '{row['table_1']}' connects to table '{row['table_2']}' by joining "
            f"'{row['table_1']}.{row['table_1_column']}' with '{row['table_2']}.{row['table_2_column']}'."
        )
        descriptions.append(sentence)
    return descriptions

def describe_tables(tables_df: pd.DataFrame) -> List[str]:
    """
    Converts the tables DataFrame into human-readable descriptions.

    Returns:
        A list of strings, where each string is a description of a table.
    """
    descriptions = []
    for _, row in tables_df.iterrows():
        # Formats a description for each table
        description = f"Table '{row['name']}': {row['description']}"
        descriptions.append(description)
    return descriptions

def describe_columns(columns_df: pd.DataFrame) -> str:
    """
    Converts the columns DataFrame into a structured, human-readable text block.

    Returns:
        A single string that describes the columns for each table.
    """
    full_description = "Column Details for Each Table:\n"
    # Group the DataFrame by table name to process each table's columns together
    for table_name, group in columns_df.groupby("table_name"):
        full_description += f"\n--- Table: {table_name} ---\n"
        for _, row in group.iterrows():
            # Add a formatted line for each column's name and description
            full_description += f"- {row['column_name']}: {row['column_description']}\n"
    return full_description


def get_scan_results(token,scan_id):
    """
    Retrieves the results of the data scan.
    """

    url = f"{BASE_URL_FOR_DATAPLEX_SCAN}/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}?view=FULL"
    headers = {"Authorization": f"Bearer {token}"}
    try:
      response = requests.get(url, headers=headers)
      response.raise_for_status()
      return response.json()

    except Exception as e:
      error_message = f"An error occurred in get_scan_results: {e}"
      return error_message


def update_bigquery_metadata(
    project_id: str,
    dataset_id: str,
    new_description: str,
    table_id: str = None,
    column_name: str = None
):
  """
  Updates the description of a BigQuery dataset, table, or column.
  """
  try:
    client = bigquery.Client(project=project_id)

    if table_id and column_name:
      # Update a column's description
      dataset_ref = client.dataset(dataset_id)
      table_ref = dataset_ref.table(table_id)
      table = client.get_table(table_ref)

      new_schema = []
      column_found = False
      for field in table.schema:
        if field.name == column_name:
          column_found = True
          # Recreate the SchemaField with the new description
          new_field = field.to_api_repr()
          new_field['description'] = new_description
          new_schema.append(bigquery.SchemaField.from_api_repr(new_field))
        else:
          new_schema.append(field)

      if not column_found:
        print(f"Error: Column '{column_name}' not found in table '{table_id}'.")
        return

      table.schema = new_schema
      client.update_table(table, ["schema"])
      print(f"Successfully updated description for column: {column_name} in table {project_id}.{dataset_id}.{table_id}")

    elif table_id:
      # Update a table's description
      dataset_ref = client.dataset(dataset_id)
      table_ref = dataset_ref.table(table_id)
      table = client.get_table(table_ref)
      table.description = new_description
      client.update_table(table, ["description"])
      print(f"Successfully updated description for table: {project_id}.{dataset_id}.{table_id}")

    else:
      # Update a dataset's description
      dataset_ref = client.dataset(dataset_id)
      dataset = client.get_dataset(dataset_ref)
      dataset.description = new_description
      client.update_dataset(dataset, ["description"])
      print(f"Successfully updated description for dataset: {project_id}.{dataset_id}")

  except NotFound as e:
    print(f"Error: Resource not found. Please check your IDs. Details: {e}")
  except Exception as e:
    print(f"An error occurred: {e}")

def get_dataset_tables(dataset_id):
  """
  Fetches a list of tables within a specified BigQuery dataset.
  This function initializes a BigQuery client for a predefined project
  and retrieves an iterator for the tables in the given dataset.
  """
  client = bigquery.Client(project=f"{PROJECT_ID}")
  tables = client.list_tables(dataset_id)
  return tables


def upload_string_as_file_to_gcs(bucket_name, blob_name, content_string) -> dict:
    """Uploads a string to a Google Cloud Storage blob."""

    try:
      # Initialize a client
      storage_client = storage.Client()

      # Get the bucket and define the blob
      bucket = storage_client.bucket(bucket_name)
      blob = bucket.blob(blob_name)

      # Upload the string data
      blob.upload_from_string(content_string, content_type="text/plain")

    except Exception as e:
        error_message = f"An error occurred while creating/persisting metadata for grounding file to GCS: {e}"
        return {"status": "error", "error": error_message}

    return {"status": "success", "detail": f"Persisted the agentic grounding string as a file to GCS - gs://{bucket_name}/{blob_name}"}



def generate_metadata_grounding_file(source_dataset_id,metadata_dataset_id,bucket_name, file_name) -> dict:
    """
    Generates metadata for agentic grounding off of the Dataplex scans persists it to a markdown file in GCS
    """

    agent_grounding_file = file_name
    agent_grounding_content = ""

    # Generate auth token
    token = get_auth_token()

    try:

        # Persist the latest Dataplex scan data to a separate dataset for use for agentic grounding
        persist_dataplex_scan_output_to_bq_tables(source_dataset_id,metadata_dataset_id)

        # Generate agentic grounding content as dataframes that can then be written to a file
        description_df = read_bigquery_table(PROJECT_ID,metadata_dataset_id, "dataset_description")
        relationships_df = read_bigquery_table(PROJECT_ID,metadata_dataset_id, "dataset_table_relationships")
        tables_df = read_bigquery_table(PROJECT_ID,metadata_dataset_id, "table_descriptions")
        columns_df = read_bigquery_table(PROJECT_ID,metadata_dataset_id, "table_column_descriptions")

        # Create the content to write to file
        agent_grounding_content += "Description of the BigQuery dataset:"
        agent_grounding_content += "\n"
        agent_grounding_content = describe_dataset(description_df)
        agent_grounding_content += "\n"
        agent_grounding_content += "------------------------------------"
        agent_grounding_content += "\n"
        agent_grounding_content += "Description of the tables in the same dataset:"
        agent_grounding_content += "\n"
        agent_grounding_content += "\n".join(describe_tables(tables_df))
        agent_grounding_content += "\n"
        agent_grounding_content += "------------------------------------"
        agent_grounding_content += "\n"
        agent_grounding_content += "The relationships between tables in the same dataset:"
        agent_grounding_content += "\n"
        agent_grounding_content += "\n".join(describe_relationships(relationships_df))
        agent_grounding_content += "\n"
        agent_grounding_content += "------------------------------------"
        agent_grounding_content += "Description of the columns of the tables in the same dataset:"
        agent_grounding_content += "\n"
        agent_grounding_content += describe_columns(columns_df)
        agent_grounding_content += "------------------------------------"

        # Write to file to GCS
        upload_string_as_file_to_gcs(bucket_name, agent_grounding_file, agent_grounding_content)
    except Exception as e:
        error_message = f"An error occurred while creating/persisting metadata for grounding file to GCS: {e}"
        return {"status": "error", "error": error_message}

    return {"status": "success", "response": f"Completed persisting metadata for grounding file to GCS"}


def persist_dataplex_scan_output_to_bq_tables(source_dataset_id,metadata_dataset_id) -> dict:
    """
    Saves to BigQuery the Data Insights scans (dataset and table documentation scans)
    for a specified dataset.
    """

    token = get_auth_token()
    source_dataset_resource = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{source_dataset_id}"
    knowledge_scan_id = sanitize_string_with_hyphens(source_dataset_id + "-dataset-documentation-scan")
    table_scan_suffix = "table-documentation-scan"


    try:

        relationships = {}
        details = {}

        # Fetch the dataset documentation scan - this includes table documentation scan as well
        results = get_scan_results(token, knowledge_scan_id)

        # Part 1: Parse the dataset description
        datasetDescription = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("overview")
        details["Description"] = datasetDescription

        data = {
                "dataset_description": datasetDescription
        }

        if datasetDescription is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_description")
            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_description", data)
            update_bigquery_metadata(PROJECT_ID, source_dataset_id, datasetDescription)


        # Part 2: Parse the table relationships
        schemaRelationships = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("schemaRelationships")
        details["Relationships"] = schemaRelationships
        if schemaRelationships is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_table_relationships")

        # Use a 'for' loop to iterate over each element in the list
        for i, relationship in enumerate(schemaRelationships):
            # Now 'relationship' is one of the dictionaries from the list

            # Safely access the data inside each dictionary
            join_type = relationship.get("type", "Unknown Type").replace("SCHEMA_JOIN", "JOIN")

            # Reset
            left_table = "N/A"
            left_column = "N/A"
            right_table = "N/A"
            right_column = "N/A"

            # Access elements of interest
            left_table_fqn = relationship.get("leftSchemaPaths", []).get("tableFqn", {})
            left_table_resource_uri_parts = left_table_fqn.split("/")
            left_table=left_table_resource_uri_parts[8]
            left_table_column = relationship.get("leftSchemaPaths", []).get("paths", {})[0]

            right_table_fqn = relationship.get("rightSchemaPaths", []).get("tableFqn", {})
            right_table_resource_uri_parts = right_table_fqn.split("/")
            right_table=  right_table_resource_uri_parts[8]
            right_table_column = relationship.get("rightSchemaPaths", []).get("paths", {})[0]

            row_data = {
                "table_1": left_table,
                "table_1_column": left_table_column,
                "table_2": right_table,
                "table_2_column": right_table_column,
                "join_type": join_type
            }

            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.dataset_table_relationships", row_data)



        # Part 3: Parse the table documentation scan payload for table descriptions
        tableResults = results.get("dataDocumentationResult", {}).get("datasetResult", {}).get("tableResults", {})
        details["tableResults"] = tableResults
        if tableResults is not None:
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.table_descriptions")
            truncate_bigquery_table(f"{PROJECT_ID}.{metadata_dataset_id}.table_column_descriptions")

        for i, tableResult in enumerate(tableResults):
          table_nm=""
          table_description=""

          table_fqn=tableResult.get("name", "")
          table_resource_uri_parts = table_fqn.split("/")
          table_nm=table_resource_uri_parts[8]
          table_description=tableResult.get("overview", "")

          row_data = {
                "name": table_nm,
                "description": table_description,
            }

          # Persist table description
          write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.table_descriptions", row_data)


          # Part 4: Parse the table documentation scan payload for table columns and descriptions
          tableColumnResults = tableResult.get("schema", {}).get("fields", [])
          details["tableColumnResults"] = tableColumnResults

          for i, tableColumnResult in enumerate(tableColumnResults):

            table_column_nm="None"
            table_column_description="None"

            table_column_nm=tableColumnResult.get("name")
            table_column_description=tableColumnResult.get("description")

            row_data = {
                  "table_name": table_nm,
                  "column_name": table_column_nm,
                  "column_description": table_column_description,
              }

            # Persist table column descriptions
            write_dict_to_bigquery(f"{PROJECT_ID}.{metadata_dataset_id}.table_column_descriptions", row_data)


    except Exception as e:
        error_message = f"An error occurred during metadata generation: {e}"
        # Return a dictionary with an error status for ADK
        return {"status": "error", "error": error_message}

    return {"status": "success",
            "response": f"Successfully saved the data scans for the dataset to BigQuery tables.",
            "details": details
            }

def run_scans_and_persist_metadata_to_file(DATASET_ID: str):
  METADATA_BUCKET = f"rscw-workshop-fridge-stage-{PROJECT_NBR}"
  SOURCE_BQ_DATASETS = DATASET_ID

  # Execute table documentation scan
  execution_results=execute_table_documentation_scan_for_a_dataset(DATASET_ID)
  print(f"Table documentation scan results: {execution_results}")
  if execution_results.startswith("ERROR"):
    raise ValueError(f"An error occurred during table documentation scan runs for dataset {DATASET_ID}. Stopping execution. The specific error is: {execution_results}")
  else:
    print(f"Successfully ran table documentation scans for all tables and views in the dataset {DATASET_ID}")

  # Execute dataset documentation scan
  execution_results=execute_dataset_documentation_scan(DATASET_ID)
  print(f"Dataset documentation scan results: {execution_results}")
  if execution_results.startswith("ERROR"):
    raise ValueError(f"An error occurred during dataset documentation scan runs for dataset {DATASET_ID}. Stopping execution. The specific error is: {execution_results}")
  else:
    print(f"Successfully ran dataset documentation scans for the dataset {DATASET_ID}")


  if DATASET_ID=="rscw_fridge_ds":
    METADATA_BQ_DATASET = "rscw_fridge_metadata_ds"
    METADATA_GROUNDING_FILE_NAME = "frige-metadata-for-agent-grounding.md"
  elif DATASET_ID=="rscw_fridge_forecast_ds":
    METADATA_BQ_DATASET = "rscw_fridge_forecast_metadata_ds"
    METADATA_GROUNDING_FILE_NAME = "frige-forecast-metadata-for-agent-grounding.md"


  # Persist metadata generated by Data Insights scans to GCS for use by agents
  print("Starting generation of metadata grounding file")
  generate_metadata_grounding_file(DATASET_ID,METADATA_BQ_DATASET,METADATA_BUCKET,METADATA_GROUNDING_FILE_NAME)
  print(f"Successfully generated the metadata grounding file ({METADATA_GROUNDING_FILE_NAME}) for use by agents and saved it to the bucket {METADATA_BUCKET}")


## 5. Run the scans for a specific dataset

In [7]:
# Modify this for the specific dataset you want to run scans for
# Dependencies:
# 1. The metadata dataset needs to exist
# 2. The tables for persisting metadata from the Data Insights scans need to exist
# 3. The GCS bucket to store the metadata export needs to exist

# Uncomment the dataset you want to generate insights for
#DATASET_ID="rscw_fridge_ds"
DATASET_ID="rscw_fridge_forecast_ds"


run_scans_and_persist_metadata_to_file(DATASET_ID)

aggr_sales_by_item_vw

Calling API: https://dataplex.googleapis.com/v1/projects/data-insights-quickstart/locations/us-central1/dataScans?dataScanId=fridge-aggr-sales-by-item-vw-table-documentation-scan
Request Body: {
  "displayName": "fridge-aggr-sales-by-item-vw-table-documentation-scan",
  "type": "DATA_DOCUMENTATION",
  "dataDocumentationSpec": {},
  "data": {
    "resource": "//bigquery.googleapis.com/projects/data-insights-quickstart/datasets/rscw_fridge_forecast_ds/tables/aggr_sales_by_item_vw"
  },
  "executionSpec": {
    "trigger": {
      "onDemand": {}
    }
  }
}
HTTP error occurred: 409 Client Error: Conflict for url: https://dataplex.googleapis.com/v1/projects/data-insights-quickstart/locations/us-central1/dataScans?dataScanId=fridge-aggr-sales-by-item-vw-table-documentation-scan
Response Body: {
  "error": {
    "code": 409,
    "message": "Resource 'projects/data-insights-quickstart/locations/us-central1/dataScans/fridge-aggr-sales-by-item-vw-table-documentation-scan' 